# Part 3: Topics

An entity graph shows how specific things are connected. **Topics** group related chunks so you can explore a broader area of the documents.

For this lab, the Foundry IQ plugin generated the topics ahead of time by clustering the chunk embeddings into a two-level hierarchy. Each topic has a label, a description, a representative vector, and a list of member chunks. Six terminal topics retain native representative vectors; two nonterminal parents retain previously derived centroids. Labels and descriptions were added from the fixed member chunks, not generated by the clustering binary.

**🎯 Mission**
- Read the topics manifest to see how much of the document set the topics cover.
- Browse the topic hierarchy.
- Pick a chunk and find its nearest neighbors in two ways: directly, and through its nearest topic.
- Query the sitemap with GQL, then use its inline-ontology knowledge source from a knowledge base.

You browsed these topics in the graph explorer's **Topics** tab. Now you'll query them with code.

## Step 1: Load your settings and connect to topics and PDF chunks

`cocoarynth-topics` contains the vectors, labels, descriptions, memberships, and sitemap edges together. `cocoarynth-chunks` contains the original PDF chunks and embeddings. No separate topic-vector, topic-source, or sitemap index is needed.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
API_VERSION = "2026-11-01-preview"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

In [ ]:
import json
import textwrap

from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery

TOPICS_INDEX = "cocoarynth-topics"
SOURCES_INDEX = "cocoarynth-chunks"
TOPICS_KNOWLEDGE_SOURCE = f"{TOPICS_INDEX}-knowledge-source"

topics_client = SearchClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, TOPICS_INDEX, search_credential, api_version=API_VERSION
)
sources_client = SearchClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, SOURCES_INDEX, search_credential, api_version=API_VERSION
)
for index_name, client in ((TOPICS_INDEX, topics_client), (SOURCES_INDEX, sources_client)):
    print(f"{index_name}: {client.get_document_count()} documents")

## Step 2: Read the topics manifest

The topics index stores several kinds of records side by side, and each document has a `kind` field that says which kind it is. The Foundry IQ plugin stores one manifest document in each topics index it builds, so the cell below filters on `kind eq 'manifest'`. The manifest records how the topics were built, so it's the place to check how much of the document set the topics cover. It also has the generation ID that later cells use to filter the topic records.

Here, 97 of 106 chunks were assigned to a topic and 9 were left out, the same numbers you saw in the graph explorer. The manifest also breaks coverage down by level: all 97 assigned chunks belong to a broad topic (level 1), but only 61 also belong to a narrower child topic (level 0), because two broad topics have no children.

In [ ]:
manifest_row = list(topics_client.search(search_text="*", filter="kind eq 'manifest'", select=["payload"], top=1))[0]
manifest = json.loads(manifest_row["payload"])
GENERATION = manifest["generation_id"]

for key in (
    "source_document_count",
    "assigned_node_count",
    "unassigned_source_document_count",
    "node_coverage",
    "vector_coverage",
    "label_stage",
    "assigned_chunks_by_level",
    "terminal_community_count",
):
    print(f"{key:<34} {manifest[key]}")

## Step 3: Query the topic hierarchy

This is the same hierarchy you browsed in the graph explorer. Here you'll see how to query it yourself, using two more kinds of records:

* `kind eq 'representative'` returns the topics. Each one has a label, a description, a member count, a `level` (1 for broad topics, 0 for narrower child topics), and a `parent_id` that links a child topic to its parent.
* `kind eq 'membership'` returns which chunks belong to a topic. Filter on `community_id` to get one topic's member chunks, then look those chunks up in `cocoarynth-chunks`.

Both filters also include the generation ID from the manifest, so you only get records from the current build of the topics.

A parent-child topic link isn't a business relationship like a farm supplying a lot: it only means the chunks are similar. Some broad topics are terminal, meaning the clustering didn't split them into children.

In [ ]:
generation_filter = f"generation_id eq '{GENERATION}'"
representatives = list(topics_client.search(
    search_text="*",
    filter=f"kind eq 'representative' and {generation_filter}",
    select=["community_id", "parent_id", "level", "member_count", "label", "description", "is_terminal"],
    top=1000,
))


def sample_titles(community_id, count=3):
    members = topics_client.search(
        search_text="*",
        filter=f"kind eq 'membership' and community_id eq '{community_id}' and {generation_filter}",
        select=["chunk_id"],
        top=count,
    )
    return [sources_client.get_document(member["chunk_id"], selected_fields=["title"])["title"] for member in members]


for parent in sorted((r for r in representatives if r["level"] == 1), key=lambda r: r["community_id"]):
    print(f"{parent['label']} ({parent['member_count']} chunks; terminal={parent['is_terminal']})")
    print(f"  {parent['description']}")
    for child in sorted((r for r in representatives if r["parent_id"] == parent["community_id"]), key=lambda r: r["community_id"]):
        print(f"  └─ {child['label']} ({child['member_count']} chunks): {', '.join(sample_titles(child['community_id']))}")

## Step 4: Pick a chunk to start from

Find a chunk that mentions Madagascar bean lot MG26-L02 and reuse its stored embedding. This asks "what's near this chunk?" rather than embedding a new question, so no embedding model is called.

In [ ]:
start_chunk = list(sources_client.search(
    search_text='"MG26-L02"',
    search_fields=["chunk"],
    select=["chunk_id", "title", "blob_path", "chunk", "text_vector"],
    top=1,
))[0]
query_vector = start_chunk["text_vector"]
print(f"{start_chunk['title']} ({start_chunk['blob_path']})")
print(textwrap.indent(textwrap.shorten(start_chunk["chunk"], 400), "    "))

## Step 5: Find the nearest chunks directly

Run a vector search over all source chunks using the starting chunk's embedding. The first result is usually the starting chunk itself.

In [ ]:
direct = list(sources_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="text_vector", k_nearest_neighbors=8, exhaustive=True)],
    select=["chunk_id", "title", "blob_path"],
    top=8,
))

for rank, result in enumerate(direct, 1):
    print(f"{rank}. {result['title']} ({result['blob_path']})")

## Step 6: Find the nearest topic, then search within it

Now route the search through a topic:

1. Find the terminal topic whose representative vector is closest to the starting chunk's embedding, by filtering on `kind eq 'representative' and is_terminal eq true`. That includes terminal roots at level 1, not just children at level 0.
2. Look up that topic's member chunks, by filtering on `kind eq 'membership'` and the topic's `community_id`.
3. Run the same vector search, filtered to just those members.

The topic narrows the set of candidate chunks. That can help focus results, but it can also miss relevant chunks in other topics or in the chunks that weren't assigned to any topic.

In [ ]:
nearest_topic = list(topics_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="centroid", k_nearest_neighbors=1, exhaustive=True)],
    filter=f"kind eq 'representative' and is_terminal eq true and {generation_filter}",
    select=["community_id", "member_count", "label", "level"],
    top=1,
))[0]
member_ids = [
    member["chunk_id"]
    for member in topics_client.search(
        search_text="*",
        filter=f"kind eq 'membership' and community_id eq '{nearest_topic['community_id']}' and {generation_filter}",
        select=["chunk_id"],
        top=1000,
    )
]
routed = list(sources_client.search(
    vector_queries=[VectorizedQuery(vector=query_vector, fields="text_vector", k_nearest_neighbors=8, exhaustive=True)],
    filter=f"search.in(chunk_id, '{','.join(member_ids)}', ',')",
    select=["chunk_id", "title", "blob_path"],
    top=8,
))

print(f"Nearest topic: {nearest_topic['label']} (level {nearest_topic['level']}; {len(member_ids)} member chunks)")
print(f"Starting chunk is in this topic: {start_chunk['chunk_id'] in member_ids}\n")
for rank, result in enumerate(routed, 1):
    print(f"{rank}. {result['title']} ({result['blob_path']})")

## Step 7: Compare the two approaches

The direct search ran one vector query. The topic-routed search also looked up the nearest topic and its members.

A chunk that appears in only one list wasn't returned by the other search. It's still in the index.

In [ ]:
direct_ids = {result["chunk_id"] for result in direct}
routed_ids = {result["chunk_id"] for result in routed}

print(f"Direct search:        {len(direct)} chunks (searched all chunks)")
print(f"Topic-routed search:  {len(routed)} chunks (searched {len(member_ids)} topic members)")
print(f"\nIn both lists:        {len(direct_ids & routed_ids)}")
print(f"Only in direct list:  {len(direct_ids - routed_ids)}")
print(f"Only in topic list:   {len(routed_ids - direct_ids)}")

## Step 8: Query the sitemap with GQL

The topic index also has a graph profile. Its navigation relationships are `Topic → hasSubtopic → Topic`, `Topic → hasMember → TextChunk`, and `Document → hasTextChunk → TextChunk`.

Run a GQL join from a broad topic through its children to their PDF documents. The query is bounded to 50 rows so the next step can choose PDFs from each child topic. This is a query over the sitemap, not another nearest-topic vector lookup. Membership means similar content, not a business relationship such as a farm supplying a lot.

In [ ]:
from azure.search.documents.models import GraphQueryProvenanceParameters

parent = next(topic for topic in representatives if not topic["is_terminal"])
PARENT_TOPIC_ID = f"urn:cocoarynth:Topic:{parent['community_id']}"
SITEMAP_GQL = f"""
MATCH (parent:`Topic`)-[:`hasSubtopic`]->(topic:`Topic`)
      -[:`hasMember`]->(chunk:`TextChunk`)<-[:`hasTextChunk`]-(document:`Document`)
WHERE parent.id = '{PARENT_TOPIC_ID}'
RETURN DISTINCT parent.name AS parent, topic.name AS topic, document.name AS document
LIMIT 50
"""
sitemap_result = topics_client.graph(
    gql=SITEMAP_GQL,
    include_statistics=True,
    provenance=GraphQueryProvenanceParameters(kind="ids", max_evidence_per_row=8),
)
for row in sitemap_result.value:
    print(f"{row['parent']} > {row['topic']}: {row['document']}")
evidence_ids = {
    item.chunk_id
    for row in sitemap_result.value if row.provenance is not None
    for item in row.provenance.evidence or [] if item.chunk_id
}
evidence = sources_client.get_document(
    sorted(evidence_ids)[0], selected_fields=["title", "blob_path", "chunk"]
)
print(f"\nExample PDF evidence: {evidence['blob_path']}")

## Step 9: Use the graph-enabled topic knowledge source

Setup created `cocoarynth-topics-knowledge-source` over the same topic index. Its `graphOntology` contains the inline JSON-LD serialization of the checked-in [sitemap OWL](../../data/graph-data/cocoarynth-sitemap.owl). Without that ontology, a search-index knowledge source would not expose the graph search tool.

![Diagram: cocoarynth-topics-knowledge-base uses cocoarynth-topics-knowledge-source, which searches the cocoarynth-topics index and carries the cocoarynth-sitemap.owl ontology inline. The topics index links to chunks in cocoarynth-chunks through hasMember and hasTextChunk.](../../img/diagrams/notebook-03.svg)

Create a knowledge base over this source and ask it to navigate the sitemap. The request includes the GQL pattern you just verified so the model does not have to infer the join's edge directions. As in Part 2, instructions encourage graph search but cannot force the model's choice. You should see a `graphSearch` step printed below the answer. If you don't, the planner used text search this time, so rerun the cell.

In [ ]:
from IPython.display import Markdown, display

from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters, KnowledgeBase, KnowledgeBaseAzureOpenAIModel,
    KnowledgeBaseMicrosoftRetriever1Model, KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage, KnowledgeBaseMessageTextContent, KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalLowReasoningEffort, KnowledgeRetrievalOutputMode,
    SearchIndexKnowledgeSourceParams,
)

index_client = SearchIndexClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, search_credential, api_version=API_VERSION
)
topic_source = index_client.get_knowledge_source(TOPICS_KNOWLEDGE_SOURCE).as_dict()
topic_ontology = topic_source["searchIndexParameters"].get("graphOntology", {})
print(f"Inline sitemap ontology: {topic_ontology['name']}")

TOPICS_KB_NAME = "cocoarynth-topics-knowledge-base"
index_client.create_or_update_knowledge_base(KnowledgeBase(
    name=TOPICS_KB_NAME,
    description="Navigate the computed Cocoarynth PDF sitemap using its inline OWL schema",
    knowledge_sources=[KnowledgeSourceReference(name=TOPICS_KNOWLEDGE_SOURCE)],
    models=[
        KnowledgeBaseMicrosoftRetriever1Model(),
        KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
            resource_url=AZURE_OPENAI_ENDPOINT, deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
            model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME, api_key=AZURE_OPENAI_KEY,
        )),
    ],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Use returned sitemap evidence only, name the PDF documents, and cite reference IDs.",
))
topics_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=TOPICS_KB_NAME,
    credential=search_credential,
    api_version=API_VERSION,
)
topic_result = topics_kb_client.retrieve(KnowledgeBaseRetrievalRequest(
    messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(
        text=f"Use the sitemap graph to list subtopics under topic {PARENT_TOPIC_ID}, "
             "and name two PDF documents in each subtopic.\n"
             "Use the verified GQL pattern below to retrieve the child-topic and "
             "PDF document names, then choose two returned PDFs per child topic:\n"
             f"{SITEMAP_GQL}"
    )])],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    include_activity=True,
    knowledge_source_params=[SearchIndexKnowledgeSourceParams(
        knowledge_source_name=TOPICS_KNOWLEDGE_SOURCE, include_references=True,
        include_reference_source_data=True, always_query_source=True,
    )],
))
display(Markdown(topic_result.response[0].content[0].text))
topic_graph_steps = [
    step.as_dict() for step in topic_result.activity or []
    if step.as_dict().get("type") == "graphSearch"
]
for step in topic_graph_steps:
    print(json.dumps(step, indent=2))

## Choose the right approach for the question

| Need | Good starting point | Tradeoff |
| --- | --- | --- |
| Ground a direct question in original documents | Standard knowledge base | The model plans queries, picks sources, and writes a cited answer |
| Retrieve chunks quickly | Microsoft Retriever 1 (MR1) | Faster planning without a chat completion model, but it searches every source and returns chunks, not an answer |
| Follow explicit relationships across records | Entity graph and GQL | Inspectable joins and provenance, but depends on the quality of the stored relationships |
| Explore groups of related chunks | Topics | Labeled organization and sitemap GQL, but similarity is not a business fact and coverage is partial |

Think about the three questions from this lab: the organic seal, the farms behind batch EC26-0811, and the Madagascar bean lot. Which approach would you start with for each, and why?

## ✅ Mission complete

**What you did:**

* ✓ **Manifest**: Read the topics manifest to see which chunks the topics cover and which they leave out.
* ✓ **Topic hierarchy**: Browsed the parent and child topics with their labels and descriptions.
* ✓ **Topic-routed search**: Compared nearest-neighbor search across all chunks with search through the nearest topic.
* ✓ **Sitemap**: Queried the sitemap with GQL and retrieved through its inline-ontology knowledge source.

➡️ Return to lab instructions.